# 05 — Population statistics

Computes the four key statistics on the canonical 1,017-neuron population: scale–elongation coupling, spatial autocorrelation of elongation, cascade-smoothing criterion, and m=1 simple-cell parameter distributions.

In [ ]:
from pathlib import Path
import sys, os

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT))

population_dir = REPO_ROOT / 'derived_data' / 'population'
gallery_dir    = REPO_ROOT / 'derived_data' / 'm1_cells'
review_dir     = REPO_ROOT / 'derived_data' / 'review_judgements'
DATASET_PATH   = gallery_dir / 'm1_neuron_dataset.pkl'

import inspect, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import pdist, squareform

from rf_analysis.sparse_noise import _pixel_size, fit_rf_by_order

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

STIMULUS_NAME   = 'locally_sparse_noise_4deg'
PIXEL_SIZE_DEG  = _pixel_size(STIMULUS_NAME)
SMOOTH_SIGMA_PX = inspect.signature(fit_rf_by_order).parameters['smooth_sigma'].default

SIGMA_SMOOTH  = SMOOTH_SIGMA_PX * PIXEL_SIZE_DEG
SIGMA_ELEMENT = PIXEL_SIZE_DEG / np.sqrt(12)
SIGMA_MEAS    = float(np.hypot(SIGMA_SMOOTH, SIGMA_ELEMENT))

In [ ]:
csv_files = sorted(population_dir.glob('rf_params_order_v2_container_*.csv'))

dfs = []
for f in csv_files:
    d = pd.read_csv(f)
    if 'container_id' not in d.columns:
        d['container_id'] = int(f.stem.split('_')[-1])
    dfs.append(d)
all_df = pd.concat(dfs, ignore_index=True)

for col in ['sigma', 'theta', 'kappa', 'r_squared', 'sigma_x', 'sigma_y',
            'x0', 'y0', 'phi', 'phi_confidence', 'theta_hybrid',
            'cortex_x_um', 'cortex_y_um', 'r2_m0', 'r2_m1', 'r2_m2',
            'delta_r2_vs_m0']:
    if col in all_df.columns:
        all_df[col] = pd.to_numeric(all_df[col], errors='coerce')

all_df['sigma_major'] = all_df[['sigma_x', 'sigma_y']].max(axis=1)
all_df['sigma_minor'] = all_df[['sigma_x', 'sigma_y']].min(axis=1)

judged, nb07_m1_ids = {}, set()
nb07_path = review_dir / 'manual_m_judgements.csv'
if nb07_path.exists():
    jdf = pd.read_csv(nb07_path)
    jdf = jdf[jdf['m_manual'] >= 0]
    for _, r in jdf.iterrows():
        judged[int(r['cell_id'])] = int(r['m_manual'])
        if int(r['m_manual']) == 1:
            nb07_m1_ids.add(int(r['cell_id']))

for path, label, col in [
    (review_dir  / 'nb09_review_judgements.csv',     'nb09',          'm_manual'),
    (population_dir / 'targeted_review_judgements.csv', 'targeted',      'm_manual'),
    (population_dir / 'unreviewed_m1_judgements.csv',   'unreviewed_m1', 'm_manual'),
    (population_dir / 'rescue_review_judgements.csv',   'rescue',        'm_final'),
]:
    if not path.exists(): continue
    jdf = pd.read_csv(path); jdf = jdf[jdf[col] >= 0]
    applied = skipped = 0
    for _, r in jdf.iterrows():
        cid, m_new = int(r['cell_id']), int(r[col])
        if cid in nb07_m1_ids and m_new == 0:
            skipped += 1; continue
        judged[cid] = m_new; applied += 1

all_df['m_manual']          = all_df['cell_id'].map(judged)
all_df['manually_verified'] = all_df['cell_id'].isin(judged)
all_df['m_final'] = np.where(all_df['m_manual'].notna(),
                             all_df['m_manual'],
                             all_df['derivative_order']).astype(int)

demote = ((all_df['m_final'] == 2) & (~all_df['manually_verified']) &
          (all_df['delta_r2_vs_m0'] < 0.10))
all_df.loc[demote, 'm_final'] = 0

N_POP = len(all_df)
auto = all_df['derivative_order'].value_counts().sort_index()
fin  = all_df['m_final'].value_counts().sort_index()


In [ ]:
with open(DATASET_PATH, 'rb') as f:
    dataset = pickle.load(f)
m1 = pd.DataFrame([{k: r.get(k) for k in
                    ['name', 'cell_id', 'container_id', 'sigma_deg', 'kappa',
                     'kappa_dir', 'sigma_phi_deg', 'sigma_orth_deg',
                     'r_squared', 'x0_deg']} for r in dataset])

s = m1['sigma_deg'].dropna()
C4 = dict(n=len(s), mean=s.mean(), sd=s.std(), cv=s.std()/s.mean(),
          median=s.median(), lo=s.min(), hi=s.max(),
          octaves=np.log2(s.max()/s.min()))



In [ ]:
cpl = all_df.dropna(subset=['sigma', 'kappa']).copy()

r_p,  p_p  = pearsonr(cpl['sigma'], cpl['kappa'])
r_s,  p_s  = spearmanr(cpl['sigma'], cpl['kappa'])

per = []
for cid, sub in cpl.groupby('container_id'):
    if len(sub) < 10:
        continue
    rr, pp = pearsonr(sub['sigma'], sub['kappa'])
    per.append({'container_id': cid, 'n': len(sub), 'r': rr, 'p': pp})
per = pd.DataFrame(per)


mm = m1.dropna(subset=['sigma_deg', 'kappa'])
r31, p31 = pearsonr(mm['sigma_deg'], mm['kappa'])
rho31, prho31 = spearmanr(mm['sigma_deg'], mm['kappa'])


In [ ]:
c = all_df.dropna(subset=['sigma_major', 'sigma_minor']).copy()

maj_c = np.sqrt(np.clip(c['sigma_major']**2 - SIGMA_SMOOTH**2, 0, None))
min_c = np.sqrt(np.clip(c['sigma_minor']**2 - SIGMA_SMOOTH**2, 0, None))
c['kappa_corr'] = np.where(min_c > 0, maj_c / min_c, np.nan)
c['sigma_corr'] = np.sqrt(maj_c * min_c)

subsets = {
    'all cells with major/minor':      np.ones(len(c), bool),
    'minor > sigma_smooth':            c['sigma_minor'] > SIGMA_SMOOTH,
    'minor > 1.5 sigma_smooth':        c['sigma_minor'] > 1.5 * SIGMA_SMOOTH,
    'minor > 2 sigma_smooth':          c['sigma_minor'] > 2.0 * SIGMA_SMOOTH,
}

for label, mask in subsets.items():
    sub = c[mask].dropna(subset=['kappa_corr', 'sigma_corr'])
    if len(sub) < 3: continue
    ru, _ = pearsonr(sub['sigma'], sub['kappa'])
    rc, _ = pearsonr(sub['sigma_corr'], sub['kappa_corr'])

In [ ]:
all_dist, all_dk, all_ds = [], [], []
per_container = []

for cid in sorted(all_df['container_id'].unique()):
    sub = all_df[all_df['container_id'] == cid].dropna(
        subset=['cortex_x_um', 'cortex_y_um', 'kappa', 'sigma'])
    if len(sub) < 10:
        continue
    xy = sub[['cortex_x_um', 'cortex_y_um']].values
    D  = squareform(pdist(xy))
    dK = squareform(pdist(sub['kappa'].values[:, None]))
    dS = squareform(pdist(sub['sigma'].values[:, None]))
    idx = np.triu_indices(len(sub), k=1)
    all_dist.extend(D[idx]); all_dk.extend(dK[idx]); all_ds.extend(dS[idx])

    r_dk, p_dk = pearsonr(D[idx], dK[idx])
    per_container.append({'container_id': cid, 'n': len(sub),
                          'r_dist_dk': r_dk, 'p_dist_dk': p_dk})

all_dist, all_dk, all_ds = map(np.array, (all_dist, all_dk, all_ds))
per_container = pd.DataFrame(per_container)

r_dk, p_dk = pearsonr(all_dist, all_dk)
r_ds, p_ds = pearsonr(all_dist, all_ds)



In [ ]:
def _wcov(mass, X, Y):
    W = mass.sum()
    if W < 1e-9:
        return None, None
    mx, my = (mass*X).sum()/W, (mass*Y).sum()/W
    dx, dy = X - mx, Y - my
    cov = np.array([[(mass*dx*dx).sum()/W, (mass*dx*dy).sum()/W],
                    [(mass*dx*dy).sum()/W, (mass*dy*dy).sum()/W]])
    return np.array([mx, my]), cov

def lobe_envelope(rf, thresh_frac=0.25):
    """Single-lobe envelope elongation perpendicular/along the differentiation
    axis, free of lobe-separation.  Returns (kappa_lobe, sep_px, phi_axis_deg)."""
    H, W = rf.shape
    Y, X = np.mgrid[0:H, 0:W].astype(float)
    on, off = np.clip(rf, 0, None), np.clip(-rf, 0, None)
    on  = np.where(on  > thresh_frac*on.max(),  on,  0.0)
    off = np.where(off > thresh_frac*off.max(), off, 0.0)
    c_on,  cov_on  = _wcov(on,  X, Y)
    c_off, cov_off = _wcov(off, X, Y)
    if c_on is None or c_off is None:
        return np.nan, np.nan, np.nan
    d = c_on - c_off
    sep = float(np.hypot(*d))
    if sep < 1e-6:
        return np.nan, np.nan, np.nan
    e_phi  = d / sep
    e_perp = np.array([-e_phi[1], e_phi[0]])
    m_on, m_off = on.sum(), off.sum()
    va = (m_on*(e_phi @cov_on @e_phi) + m_off*(e_phi @cov_off @e_phi)) / (m_on+m_off)
    vp = (m_on*(e_perp@cov_on @e_perp)+ m_off*(e_perp@cov_off@e_perp)) / (m_on+m_off)
    kappa_lobe = float(np.sqrt(vp/va)) if va > 1e-9 else np.nan
    return kappa_lobe, sep, float(np.rad2deg(np.arctan2(e_phi[1], e_phi[0])) % 180)

_px = {float(r['pixel_size_deg']) for r in dataset}

rows = []
for r in dataset:
    kl, sep, ax = lobe_envelope(np.asarray(r['rf_smooth'], float))
    rows.append({'name': r['name'], 'kappa_lobe': kl, 'sep_px': sep,
                 'kappa_dir': r['kappa_dir']})
lob = pd.DataFrame(rows)

kd = lob['kappa_dir'].dropna()
kl = lob['kappa_lobe'].dropna()

